In [1]:
import pandas as pd
import re
import emoji
import html
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
tqdm.pandas()

print("Library berhasil dimuat")

Library berhasil dimuat


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("rawData/dataX.csv")
df = df[['created_at', 'user_id_str', 'full_text']]
df['full_text'] = df['full_text'].fillna('')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3714 entries, 0 to 3713
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   created_at   3714 non-null   object
 1   user_id_str  3714 non-null   int64 
 2   full_text    3714 non-null   object
dtypes: int64(1), object(2)
memory usage: 87.2+ KB


In [3]:
initial_count = len(df)
df.drop_duplicates(subset=['full_text'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Data dimuat: {len(df)} tweet (dihapus {initial_count - len(df)} duplikat)")
print(f"Kolom: {df.columns.tolist()}")

Data dimuat: 1882 tweet (dihapus 1832 duplikat)
Kolom: ['created_at', 'user_id_str', 'full_text']


In [4]:
def clean_tweet_bertopic(text):
    if not isinstance(text, str):
        return ""

    # Decode HTML entities (mis. &amp; → &)
    text = html.unescape(text)

    # Normalkan emoji → teks deskriptif, lalu hapus tag-nya
    text = emoji.demojize(text, language='en')
    text = re.sub(r':[\w_]+:', ' ', text)

    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Hapus mention (@username)
    text = re.sub(r'@\w+', '', text)

    # Hapus simbol hashtag, pertahankan kata di belakangnya
    text = re.sub(r'#(\w+)', r'\1', text)

    # Hapus indikator retweet
    text = re.sub(r'^RT[\s]+', '', text)

    # Hapus artefak HTML tersisa (mis. &gt; &lt;)
    text = re.sub(r'&\w+;', '', text)

    # Hapus tanda baca, simbol, dan angka
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)

    # Bersihkan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def case_folding(text):
    if not isinstance(text, str):
        return ""
    return text.lower()

# Kolom data_cleaning: hasil bersih sebelum case folding
df['data_cleaning'] = df['full_text'].progress_apply(clean_tweet_bertopic)

# Kolom case_folding: hasil setelah diubah ke huruf kecil
df['case_folding'] = df['data_cleaning'].progress_apply(case_folding)

print("Cleaning dan case folding selesai")

100%|██████████| 1882/1882 [00:00<00:00, 1254358.83it/s]

Cleaning dan case folding selesai


In [5]:
def normalize_minimal(text):
    normalization_dict = {
        'yg'    : 'yang',
        'dg'    : 'dengan',
        'dgn'   : 'dengan',
        'ga'    : 'tidak',
        'gak'   : 'tidak',
        'tdk'   : 'tidak',
        'krn'   : 'karena',
        'sdh'   : 'sudah',
        'udh'   : 'sudah',
        'bs'    : 'bisa',
        'tp'    : 'tapi',
        'utk'   : 'untuk',
        'banget': 'sangat',
        'bgt'   : 'sangat',
        'deh'   : '',
        'sih'   : '',
        'dong'  : '',
        'ya'    : '',
        'kok'   : '',
        'gw'    : 'saya',
        'gue'   : 'saya',
        'lu'    : 'kamu',
        'loe'   : 'kamu',
        'elo'   : 'kamu',
    }

    words = text.split()
    normalized = []
    for word in words:
        replacement = normalization_dict.get(word)
        if replacement is None:
            normalized.append(word)
        elif replacement:
            normalized.append(replacement)

    return ' '.join(normalized)

# Input dari case_folding, output ke kolom normalization
df['normalization'] = df['case_folding'].progress_apply(normalize_minimal)

print("Normalisasi minimal selesai")
print(f"Ukuran kamus normalisasi: {24} istilah kritis")

100%|██████████| 1882/1882 [00:00<00:00, 187765.94it/s]

Normalisasi minimal selesai
Ukuran kamus normalisasi: 24 istilah kritis


In [6]:
print("=" * 60)
print("LAPORAN KUALITAS PREPROCESSING — DATA X (BERTopic)")
print("=" * 60)

df['orig_length']  = df['full_text'].str.len()
df['final_length'] = df['normalization'].str.len()
df['orig_words']   = df['full_text'].apply(lambda x: len(str(x).split()))
df['final_words']  = df['normalization'].apply(lambda x: len(str(x).split()))

avg_orig  = df['orig_length'].mean()
avg_final = df['final_length'].mean()
avg_orig_w  = df['orig_words'].mean()
avg_final_w = df['final_words'].mean()

print(f"\nStatistik panjang teks:")
print(f"  Rata-rata panjang asli  : {avg_orig:.1f} karakter")
print(f"  Rata-rata panjang akhir : {avg_final:.1f} karakter")
print(f"  Reduksi                 : {((avg_orig - avg_final) / avg_orig * 100):.1f}%")

print(f"\nStatistik jumlah kata:")
print(f"  Rata-rata kata asli  : {avg_orig_w:.1f}")
print(f"  Rata-rata kata akhir : {avg_final_w:.1f}")
print(f"  Reduksi              : {((avg_orig_w - avg_final_w) / avg_orig_w * 100):.1f}%")

print(f"\nDistribusi panjang teks akhir:")
print(f"  Terpendek : {df['final_length'].min()} karakter")
print(f"  Terpanjang: {df['final_length'].max()} karakter")
print(f"  Median    : {df['final_length'].median():.1f} karakter")

LAPORAN KUALITAS PREPROCESSING — DATA X (BERTopic)

Statistik panjang teks:
  Rata-rata panjang asli  : 167.5 karakter
  Rata-rata panjang akhir : 142.8 karakter
  Reduksi                 : 14.7%

Statistik jumlah kata:
  Rata-rata kata asli  : 23.9
  Rata-rata kata akhir : 22.0
  Reduksi              : 8.3%

Distribusi panjang teks akhir:
  Terpendek : 10 karakter
  Terpanjang: 288 karakter
  Median    : 137.0 karakter


In [7]:
bertopic_data = df[[
    'created_at',
    'user_id_str',
    'full_text',         # teks asli
    'data_cleaning',     # setelah cleaning
    'case_folding',      # setelah huruf kecil
    'normalization',     # setelah normalisasi kata
]].copy()

bertopic_data['preprocessing_stats'] = bertopic_data.apply(
    lambda row: f"orig:{len(row['full_text'])} -> final:{len(row['normalization'])}",
    axis=1
)

output_file = "cleanData/bertopic_ready_x.csv"
bertopic_data.to_csv(output_file, index=False, encoding='utf-8')

print(f"Data diekspor ke  : {output_file}")
print(f"Jumlah tweet siap : {len(bertopic_data)}")
print(f"Kolom             : {bertopic_data.columns.tolist()}")

Data diekspor ke  : cleanData/bertopic_ready_x.csv
Jumlah tweet siap : 1882
Kolom             : ['created_at', 'user_id_str', 'full_text', 'data_cleaning', 'case_folding', 'normalization', 'preprocessing_stats']
